In [5]:
import pandas as pd
import os

featureFile = pd.read_csv('data input/All nets.csv')
featureFile['Sub_ID'] = featureFile['Sub_ID'].str.replace('sub-','')
featureFile.rename(columns={'Sub_ID':'CCID'},inplace=True)
featureFile.head()


,CCID,Age,Sex,mean_FD,GmVol,VIS1,VIS2,VIS3,VIS4,VIS5,...,DMN39,DMN40,DMN41,DMN42,DMN43,DMN44,DMN45,DMN46,DMN47,DMN48
0,CC110033,24,1,0.175219,650110.158970,1,25,43,19,77,...,2,55,46,40,11,25,68,138,59,94
1,CC110045,24,2,0.115785,691154.742770,1,19,31,27,75,...,1,32,43,46,12,42,69,97,86,98
2,CC110087,28,2,0.216117,609016.010346,1,19,48,18,69,...,0,51,47,45,9,26,66,109,83,86
3,CC110126,22,2,0.178761,687770.298194,1,23,36,24,58,...,1,38,64,31,13,28,69,140,58,95
4,CC110174,25,2,0.162931,530704.460014,1,26,22,34,65,...,2,42,42,49,12,39,66,113,75,94


In [ ]:
# EER is processed separately because PCA needs to be computed
ct = pd.read_excel('data input/stage2cogtask.xlsx')
ct.head()

,CCID,Anger,Disgust,Fear,Happy,Sad,Surprise,AngMeanRT,DisMeanRT,FeaMeanRT,HapMeanRT,SadMeanRT,SurMeanRT
0,CC110033,80,90,85,100,100,90,3051.6,2163.8,3020.1,1958.8,1889.5,3631.8
1,CC110037,90,95,75,95,95,95,3393.3,3306.9,3771.2,2633.8,2890.9,2500.7
2,CC110045,90,95,100,100,95,90,3185.0,1523.4,2600.8,1798.7,1763.3,2174.3
3,CC110056,95,100,85,100,100,60,2613.1,1924.2,2709.1,1927.8,2146.7,2540.7
4,CC110062,100,85,75,95,100,100,2042.7,2331.8,2647.9,2338.2,2011.8,2297.7


In [ ]:
# First filter subjects, then perform PCA
EERandFeature = pd.merge(featureFile, ct, on='CCID', how='inner')
EERandFeature.head()

,CCID,Age,Sex,mean_FD,GmVol,VIS1,VIS2,VIS3,VIS4,VIS5,...,Fear,Happy,Sad,Surprise,AngMeanRT,DisMeanRT,FeaMeanRT,HapMeanRT,SadMeanRT,SurMeanRT
0,CC110033,24,1,0.175219,650110.158970,1,25,43,19,77,...,85,100,100,90,3051.6,2163.8,3020.1,1958.8,1889.5,3631.8
1,CC110045,24,2,0.115785,691154.742770,1,19,31,27,75,...,100,100,95,90,3185.0,1523.4,2600.8,1798.7,1763.3,2174.3
2,CC110087,28,2,0.216117,609016.010346,1,19,48,18,69,...,95,100,100,100,2225.7,2284.4,3020.3,1865.9,1961.5,2185.5
3,CC110126,22,2,0.178761,687770.298194,1,23,36,24,58,...,90,100,100,95,2233.4,1790.4,2267.9,1813.6,2068.1,2239.8
4,CC110174,25,2,0.162931,530704.460014,1,26,22,34,65,...,90,95,100,85,2050.4,1879.4,3159.1,1887.5,1594.2,2702.7


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=1)
pca.fit(EERandFeature.iloc[:,-6:])
# Obtain the dimensionality-reduced results of the original data; can also use new data as parameter to get reduced results
PC1ofRT = pca.transform(EERandFeature.iloc[:,-6:])
print(PC1ofRT)
# Print the variance explained by each principal component
print('='*20)
print(pca.explained_variance_ratio_)

In [ ]:
from sklearn.preprocessing import StandardScaler
# Does standardization make a difference?
scaler = StandardScaler()
X_scaled = scaler.fit_transform(EERandFeature.iloc[:,-6:])
pca = PCA(n_components=1)
X_pca = pca.fit_transform(X_scaled)
# Obtain the dimensionality-reduced results of the original data; can also use new data as parameter to get reduced results
print(X_pca)
# Print the variance explained by each principal component
print('='*20)
print(pca.explained_variance_ratio_)

In [ ]:
X_scaled

In [17]:
EERandFeature['PC1ofRT'] = X_scaled
EERandFeature.head()

,CCID,Age,Sex,mean_FD,GmVol,VIS1,VIS2,VIS3,VIS4,VIS5,...,Happy,Sad,Surprise,AngMeanRT,DisMeanRT,FeaMeanRT,HapMeanRT,SadMeanRT,SurMeanRT,PC1ofRT
0,CC110033,24,1,0.175219,650110.158970,1,25,43,19,77,...,100,100,90,3051.6,2163.8,3020.1,1958.8,1889.5,3631.8,-0.362218
1,CC110045,24,2,0.115785,691154.742770,1,19,31,27,75,...,100,95,90,3185.0,1523.4,2600.8,1798.7,1763.3,2174.3,-1.150623
2,CC110087,28,2,0.216117,609016.010346,1,19,48,18,69,...,100,100,100,2225.7,2284.4,3020.3,1865.9,1961.5,2185.5,-0.213746
3,CC110126,22,2,0.178761,687770.298194,1,23,36,24,58,...,100,100,95,2233.4,1790.4,2267.9,1813.6,2068.1,2239.8,-0.821916
4,CC110174,25,2,0.162931,530704.460014,1,26,22,34,65,...,95,100,85,2050.4,1879.4,3159.1,1887.5,1594.2,2702.7,-0.712347


In [18]:
EERandFeature.to_csv('data input/EERextra.csv',index=False)

In [24]:
taskNames = ['MotorLearning','VSTM','PicturePriming','RTchoice','RTsimple','FamousFaces','EmotionRegulation']
for t in taskNames:
    currT = pd.read_excel('data input/stage2cogtask.xlsx',sheet_name = t)
    mergedDf =pd.merge(featureFile,currT,on='CCID',how='inner')
    mergedDf.to_csv(f'data input/{t}_extra.csv',index=False)